# CEG Colab 端到端论文结果包流水线

该 Notebook 只负责 Colab 调度: 从 GitHub 拉取 CEG 代码, 从 Google Drive 读取前序输入或保存输出, 调用仓库内正式脚本, 最终把图像生成产物和论文结果包归档回 Google Drive。


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

REPO_URL = "https://github.com/RICHAAARC/CEG.git"
REPO_DIR = Path("/content/CEG")
DRIVE_ROOT = Path("/content/drive/MyDrive/CEG")
WORKSPACE = DRIVE_ROOT / "pilot_runs" / "real_pilot_input_workspace_current"
RUN_ID = WORKSPACE.name

# 正式运行配置: True 表示本 Notebook 会调用真实 SD 与 CEG watermark backend。
RUN_IMAGE_GENERATION = True
SD_MODEL_ID = "stabilityai/stable-diffusion-3.5-medium"
HF_TOKEN_ENV = "HF_TOKEN"
WATERMARK_BACKEND = "ceg_content_chain_embedding"
SEMANTIC_MASK_BACKEND = "ceg_inspyrenet_semantic_mask"
ATTESTATION_KEY_ENV = "CEG_ATTESTATION_KEY"
ATTESTATION_KEY_ID = "formal_colab_run_key"
DETECTION_FORMAL_RESULT_CLAIM = True
PREPARE_INSPYRENET_WEIGHT = True
INSPYRENET_WEIGHT_URL = "https://huggingface.co/plemeri/InSPyReNet/resolve/main/ckpt_base.pth"

# 论文结果包配置。
ATTACK_FAMILIES = "brightness_contrast,gaussian_noise,rotate,resize,jpeg"
TARGET_FPR = "0.01"
PROFILE = "paper_main_probe"
ALLOW_INCOMPLETE_PACKAGE = False
ALLOW_INVALID_ARCHIVE = False
REFRESH_STAGE_SUMMARY = True

PROMPT_PLAN = WORKSPACE / "inputs" / "prompts" / "prompt_plan.draft.json"
MODEL_CONFIG = WORKSPACE / "configs" / "model_config.draft.json"
INSPYRENET_WEIGHT_DRIVE_PATH = DRIVE_ROOT / "Models" / "inspyrenet" / "ckpt_base.pth"
INSPYRENET_CACHE_PATHS = [Path.home() / ".transparent-background" / "ckpt_base.pth", Path.home() / ".cache" / "transparent-background" / "ckpt_base.pth"]
IMAGE_OUTPUT_ROOT = WORKSPACE / "inputs" / "images"
PIPELINE_OUT = WORKSPACE / "paper_end_to_end_pipeline"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


def ensure_ceg_attestation_key() -> None:
    """在 Colab 运行时定义 CEG_ATTESTATION_KEY, 只放入环境变量, 不写入文件或日志。"""
    if os.environ.get(ATTESTATION_KEY_ENV):
        print(f"{ATTESTATION_KEY_ENV} 已存在, 不打印密钥内容。")
        return
    secret_value = None
    try:
        from google.colab import userdata  # type: ignore
        secret_value = userdata.get(ATTESTATION_KEY_ENV)
    except Exception:
        secret_value = None
    if not secret_value:
        import secrets
        secret_value = secrets.token_urlsafe(48)
        print(f"{ATTESTATION_KEY_ENV} 未在 Colab secrets 中定义, 已为本次运行生成临时密钥。")
    os.environ[ATTESTATION_KEY_ENV] = secret_value
    print(f"{ATTESTATION_KEY_ENV} configured =", bool(os.environ.get(ATTESTATION_KEY_ENV)))


def prepare_inspyrenet_weight() -> None:
    """准备 InSPyReNet 权重, 该逻辑仅属于 Colab 环境准备, 不进入 CEG 主方法。"""
    if not PREPARE_INSPYRENET_WEIGHT:
        print("PREPARE_INSPYRENET_WEIGHT = False, 跳过 InSPyReNet 权重准备。")
        return
    source_path = INSPYRENET_WEIGHT_DRIVE_PATH
    if not source_path.is_file():
        source_path.parent.mkdir(parents=True, exist_ok=True)
        download_cmd = ["python", "-c", "import urllib.request, sys; urllib.request.urlretrieve(sys.argv[1], sys.argv[2])", INSPYRENET_WEIGHT_URL, str(source_path)]
        print("Drive 中未找到 InSPyReNet 权重, 下载到:", source_path)
        subprocess.run(download_cmd, check=True)
    for cache_path in INSPYRENET_CACHE_PATHS:
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        if not cache_path.exists():
            shutil.copyfile(source_path, cache_path)
    os.environ["INSPYRENET_CKPT_PATH"] = str(source_path)
    print("InSPyReNet 权重已准备, 不写入主方法代码:", source_path)

ensure_ceg_attestation_key()
prepare_inspyrenet_weight()



In [ ]:
if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers", "transformers", "accelerate", "safetensors", "Pillow", "PyYAML", "transparent-background"], check=True)


In [ ]:
# 从 Google Drive 加载前序产物结果。正式图像生成需要 prompt plan 和 model config。
required_inputs = [WORKSPACE]
if RUN_IMAGE_GENERATION:
    required_inputs.extend([PROMPT_PLAN, MODEL_CONFIG])
else:
    required_inputs.append(IMAGE_OUTPUT_ROOT / "image_pairs.json")
missing = [str(path) for path in required_inputs if not path.exists()]
if missing:
    raise FileNotFoundError("Google Drive 工作区缺少端到端流水线输入: " + ", ".join(missing))
if RUN_IMAGE_GENERATION and HF_TOKEN_ENV not in os.environ:
    raise EnvironmentError(f"环境变量 {HF_TOKEN_ENV} 未定义, 无法从 Hugging Face 加载正式 SD 模型。")


In [ ]:
cmd = [
    sys.executable,
    "scripts/run_colab_end_to_end_paper_pipeline.py",
    "--workspace", str(WORKSPACE),
    "--drive-root", str(DRIVE_ROOT),
    "--run-id", RUN_ID,
    "--out", str(PIPELINE_OUT),
    "--prompt-plan", str(PROMPT_PLAN),
    "--model-config", str(MODEL_CONFIG),
    "--image-output-root", str(IMAGE_OUTPUT_ROOT),
    "--sd-model-id", SD_MODEL_ID,
    "--hf-token-env", HF_TOKEN_ENV,
    "--watermark-backend", WATERMARK_BACKEND,
    "--semantic-mask-backend", SEMANTIC_MASK_BACKEND,
    "--attestation-key-env", ATTESTATION_KEY_ENV,
    "--attestation-key-id", ATTESTATION_KEY_ID,
    "--attack-families", ATTACK_FAMILIES,
    "--target-fpr", TARGET_FPR,
    "--profile", PROFILE,
]
if RUN_IMAGE_GENERATION:
    cmd.append("--run-image-generation")
if DETECTION_FORMAL_RESULT_CLAIM:
    cmd.append("--detection-formal-result-claim")
if ALLOW_INCOMPLETE_PACKAGE:
    cmd.append("--allow-incomplete-package")
if ALLOW_INVALID_ARCHIVE:
    cmd.append("--allow-invalid-archive")
if REFRESH_STAGE_SUMMARY:
    cmd.append("--refresh-stage-summary")
print("运行:", " ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
manifest_path = PIPELINE_OUT / "colab_end_to_end_paper_pipeline_manifest.json"
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
print(json.dumps({
    "overall_decision": manifest.get("overall_decision"),
    "image_generation_archive_zip": manifest.get("image_generation_archive_zip"),
    "paper_results_package_root": manifest.get("paper_results_package_root"),
    "paper_pipeline_manifest": manifest.get("paper_pipeline_manifest"),
}, ensure_ascii=False, indent=2))
if manifest.get("overall_decision") != "pass":
    raise RuntimeError("端到端论文结果包流水线未通过。")

# 对本次端到端运行做正式验收。正式 GPU 运行不应开启 allow-existing-image-generation。
formal_acceptance_report = PIPELINE_OUT / "colab_end_to_end_formal_run_acceptance_report.json"
acceptance_cmd = [
    sys.executable,
    "scripts/validate_colab_end_to_end_formal_run.py",
    "--manifest", str(manifest_path),
    "--out", str(formal_acceptance_report),
    "--require-evidence",
    "--require-image-examples",
    "--require-pass",
]
subprocess.run(acceptance_cmd, check=True)
formal_acceptance = json.loads(formal_acceptance_report.read_text(encoding="utf-8"))
print(json.dumps({
    "formal_acceptance_decision": formal_acceptance.get("overall_decision"),
    "formal_acceptance_report": str(formal_acceptance_report),
}, ensure_ascii=False, indent=2))
